In [29]:
import warnings
warnings.filterwarnings('ignore')

In [30]:
import numpy as np
import pandas as pd
import geopandas as gpd

In [31]:
urb = pd.read_csv('../../2026_data/urban_rural_demo/DECENNIALDHC2020.P2-Data.csv')
urb.columns = urb.iloc[0]
urb = urb.iloc[1:]
urb.head()

,Geography,Geographic Area Name,!!Total:,!!Total:!!Urban,!!Total:!!Rural,!!Total:!!Not defined for this file,NaN
1,1000000US010010201001000,"Block 1000, Block Group 1, Census Tract 201, A...",21,21,0,0,NaN
2,1000000US010010201001001,"Block 1001, Block Group 1, Census Tract 201, A...",34,34,0,0,NaN
3,1000000US010010201001002,"Block 1002, Block Group 1, Census Tract 201, A...",29,0,29,0,NaN
4,1000000US010010201001003,"Block 1003, Block Group 1, Census Tract 201, A...",17,0,17,0,NaN
5,1000000US010010201001004,"Block 1004, Block Group 1, Census Tract 201, A...",0,0,0,0,NaN


In [32]:
assn_sheet = pd.DataFrame()

for st in ['AL', 'CA', 'FL', 'LA', 'MO', 'NC', 'OH', 'TN', 'TX', 'UT']:
    cong = pd.read_csv(f'../../2026_data/cong_block_assgns/{st}-2026-Congressional-block-assignments.csv')
    cong['state_po'] = np.full(cong.shape[0], st)
    cong = cong.rename({'District': 'seat_number'}, axis=1)
    cong['district'] = cong['state_po'] + '-' + cong['seat_number'].map(lambda x: f'0{x}' if x < 10 else str(x))
    cong['GEOID20'] = cong['GEOID20'].astype(int)
    assn_sheet = pd.concat([assn_sheet, cong], axis=0)

assn_sheet.head()

,GEOID20,seat_number,state_po,district
0,10010210001000,6,AL,AL-06
1,10010210001001,6,AL,AL-06
2,10010210001002,6,AL,AL-06
3,10010210001003,6,AL,AL-06
4,10010210001004,6,AL,AL-06


In [33]:
urb['GEOID20'] = urb['Geography'].str.replace('1000000US', '').astype(int)
urb.head(2)

,Geography,Geographic Area Name,!!Total:,!!Total:!!Urban,!!Total:!!Rural,!!Total:!!Not defined for this file,NaN,GEOID20
1,1000000US010010201001000,"Block 1000, Block Group 1, Census Tract 201, A...",21,21,0,0,NaN,10010201001000
2,1000000US010010201001001,"Block 1001, Block Group 1, Census Tract 201, A...",34,34,0,0,NaN,10010201001001


In [34]:
urb = pd.merge(left=assn_sheet, right=urb, on='GEOID20', how='left')
urb.head()

,GEOID20,seat_number,state_po,district,Geography,Geographic Area Name,!!Total:,!!Total:!!Urban,!!Total:!!Rural,!!Total:!!Not defined for this file,NaN
0,10010210001000,6,AL,AL-06,1000000US010010210001000,"Block 1000, Block Group 1, Census Tract 210, A...",22,0,22,0,NaN
1,10010210001001,6,AL,AL-06,1000000US010010210001001,"Block 1001, Block Group 1, Census Tract 210, A...",11,0,11,0,NaN
2,10010210001002,6,AL,AL-06,1000000US010010210001002,"Block 1002, Block Group 1, Census Tract 210, A...",0,0,0,0,NaN
3,10010210001003,6,AL,AL-06,1000000US010010210001003,"Block 1003, Block Group 1, Census Tract 210, A...",0,0,0,0,NaN
4,10010210001004,6,AL,AL-06,1000000US010010210001004,"Block 1004, Block Group 1, Census Tract 210, A...",0,0,0,0,NaN


In [35]:
urb.columns.values

array(['GEOID20', 'seat_number', 'state_po', 'district', 'Geography',
       'Geographic Area Name', ' !!Total:', ' !!Total:!!Urban',
       ' !!Total:!!Rural', ' !!Total:!!Not defined for this file',
       np.float64(nan)], dtype=object)

In [36]:
urb.columns = urb.columns.str.strip()
urb = urb.rename({
    '!!Total:': 'total',
    '!!Total:!!Urban': 'urban',
    '!!Total:!!Rural': 'rural'
}, axis=1)
for col in ['total', 'urban', 'rural']:
    urb[col] = urb[col].astype(int)
urb = urb.groupby(['district'])[['total', 'urban', 'rural']].sum()
urb['urban_pct'] = urb['urban'] / urb['total'] * 100
urb['rural_pct'] = urb['rural'] / urb['total'] * 100
urb.head()

,total,urban,rural,urban_pct,rural_pct
district,,,,,
AL-01,717754,490405,227349,68.324941,31.675059
AL-02,717755,404336,313419,56.333429,43.666571
AL-03,717754,350943,366811,48.894607,51.105393
AL-04,717754,238623,479131,33.245792,66.754208
AL-05,717754,461471,256283,64.293755,35.706245


In [37]:
urb.to_csv('../../2026_data/redist_seats_2026_urbrur.csv')